# WindDB2

`QSExt.Factor.WindDB2.WindDB2` 通过 SQL 连接读取 Wind 量化研究数据库。参数、通用 SQL 因子表和计算图接口与当前 QuantStudio 框架一致，返回 `Panel(items=因子, major_axis=时点, minor_axis=ID)`。

## 配置与连接

默认查找 `~/QuantStudioConfig/WindDB2Config.json`。配置键采用英文，如 `DBType`、`DBName`、`Connector`、`TablePrefix`、`DBInfoFile`、`FTArgs`。连接凭据放在本地配置中。显式传入的 `args` 覆盖配置文件，配置文件覆盖模型默认值。

默认使用 QSExt 自带的 `Resource/WindDB2Info.xlsx`，自定义 `DBInfoFile` 需要包含 `TableInfo`、`FactorInfo` 两张工作表。元数据读取时会去除重复的表名、字段，适配历史财务及成份字段类型，并修正 LOF 元数据中的表名差异。原始 Excel 不会被修改。

`TableNames` 按实际数据库中的表筛选，读取前应先连接数据库。

In [ ]:
import datetime as dt
from QSExt.Factor.WindDB2 import WindDB2

# 运行前在本地配置文件中设置连接信息。
fdb = WindDB2().connect()
table = fdb.getTable('中国A股日行情', args={'MultiMapping': False})
data = table.readData(
    factor_names=['收盘价(元)'],
    ids=['000001.SZ'],
    dts=[dt.datetime(2024, 1, 2)],
)
data.iloc[0]

## 因子表参数

表参数优先级为：`getTable(args=...)` > 表元数据 `DefaultArgs` > 库参数 `FTArgs` > 默认值。推荐使用英文参数名；旧元数据及 `getTable` 中的中文参数名会按参数标题转换，若同一层同时给出中英文键，以英文键为准。默认 `DTFmt` 和 `DateFmt` 均为 `%Y%m%d`。

支持宽表、窄表、特征表、时序表、映射表、成份表、财务表，以及分析师汇总、盈利预测明细、评级明细表。表类型使用 `TableType` 指定，采用当前框架已实现的类型名称。

### 财务表

通过 `CalcType` 选择 `最新`、`单季度` 或 `TTM`，通过 `ReportDate` 选择报告期，通过 `YearLookBack`、`PeriodLookBack` 指定回溯。年数和期数同时非零时沿用旧 Wind 逻辑，优先按年数回溯。`ReportType` 默认保留 `408001000`、`408004000`、`408005000`。

报表含类型字段时，保留原脚本的公告日期口径：`408004000` 使用本表公告日期，其余类型优先采用实际披露日期。`ExprFactor`、`NoticeFactor` 分别指定业绩快报和预告字段；预告值保留万元转元的乘数 `10000`。只使用当前时点已经公告的数据。

### 分析师表

汇总表使用 `CalcType=FY0/FY1/FY2/Fwd12M`、`LookBack` 和 Wind 周期代码 `Period`。FY 年度在上一年年报实际公告当日切换；缺少该公告记录时按尚未披露处理。`Fwd12M` 保留按剩余天数加权的原有算法。

明细表使用 `Operator`、`ModelArgs`、`AdditionalFields`、`Deduplication`、`Period`、`DataType`，盈利预测明细另有 `ForwardYears`。默认按研究机构保留窗口内最新日期的记录。算子签名为 `operator(table, date, iid, x, args)`：评级的 `x` 是 DataFrame，盈利预测的 `x` 是与 `ForwardYears` 对应的 DataFrame 列表。原始数据的 ID 列统一为 `QS_ID`；时间列保留 `日期`，日期值保留 Wind 的 `%Y%m%d` 字符串。默认算子返回缺失值，使用明细表时应传入业务算子。

分析师数据按日计算，再恢复请求时点的顺序，支持同一天的多个盘中时点。预测原始数据和年报公告数据通过当前 `FactorCache` 分别缓存，支持按 ID 分配给进程。

In [ ]:
financial_table = fdb.getTable('中国A股利润表', args={'CalcType': 'TTM'})
consensus_table = fdb.getTable(
    '中国A股盈利预测汇总',
    args={'CalcType': 'FY1', 'Period': '263001000', 'LookBack': 180},
)

## 迁移说明与验证

构造参数从旧的 `sys_args` 改为 `args`。因子表参数在 `getTable` 时设置，因子通过该表的 `getFactor` 创建。`getTradeDay`、`getStockID(index_id=...)`、`getMutualFundID`、`getFutureID`、`getFutureCode` 和 `getOptionID` 保留原有 Wind 查询语义及代码格式，不自动转换为聚源证券分类。

回归测试位于 `tests/test_Factor_WindDB2.py`。在 QSExt 根目录运行 `python -m unittest discover -s tests -p test_Factor_WindDB2.py`，覆盖元数据构造、财务计算、公告边界、分析师预测、Feather 缓存及计算图读取。测试使用模拟查询结果，不连接真实 Wind 数据库；真实数据库的表结构、权限和方言兼容性需要在部署环境中验证。